# 01. Baseline: як декодери читають чисті коди

**Тиждень 1.** Перш ніж щось псувати й покращувати, перевіряємо, що всі три декодери
читають чисті коди, і розуміємо, що саме вони повертають.

- `opencv`: `cv2.barcode.BarcodeDetector`, лише EAN/UPC
- `wechat`: `cv2.wechat_qrcode_WeChatQRCode`, лише QR
- `zxing`: `zxing-cpp`, усі формати (EAN-13, Code128, QR)

In [ ]:
import sys
from pathlib import Path

# ноутбук лежить у notebooks/, корінь проєкту на рівень вище
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
RESULTS = PROJECT_ROOT / "results"
RESULTS.mkdir(exist_ok=True)

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 160)

In [ ]:
from src.data import generate_synthetic, render_code, camera_look, load_own_photos
from src.decoders import DECODERS, decoders_for

N_PER_FORMAT = 10   # кодів кожного формату
samples = generate_synthetic(n_per_format=N_PER_FORMAT, seed=42)
print("Зразків:", len(samples))

## Як виглядають синтетичні коди

In [ ]:
examples = [samples[0], samples[N_PER_FORMAT], samples[2 * N_PER_FORMAT]]
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, sample in zip(axes, examples):
    ax.imshow(sample["image"], cmap="gray", vmin=0, vmax=255)
    ax.set_title(sample["format"] + ": " + sample["text"], fontsize=9)
    ax.axis("off")
plt.show()

## Що повертає кожен декодер

Результат кожного декодера у `src/decoders.py` зведено до одного формату:
список словників `{"text": ..., "format": ...}`. Порожній список означає «нічого не прочитано».

In [ ]:
sample = samples[0]
for name in decoders_for(sample["format"]):
    print(name, "->", DECODERS[name](sample["image"]))

## Частка прочитаних чистих кодів

In [ ]:
rows = []
for sample in samples:
    for name in decoders_for(sample["format"]):
        results = DECODERS[name](sample["image"])
        correct = False
        for result in results:
            if result["text"] == sample["text"]:
                correct = True
        rows.append({"format": sample["format"], "decoder": name, "correct": correct})

clean_table = pd.DataFrame(rows)
clean_table.pivot_table(index="format", columns="decoder", values="correct", aggfunc="mean")

## Несподіванка: ідеально різкий код OpenCV не бачить

Знайдено на початку проєкту. Тому `generate_synthetic` пропускає кожен код через
`camera_look` (реальні рівні 30/230, легке розмиття, шум).

In [ ]:
perfect = render_code(samples[0]["text"], "EAN13", module_px=4)
camera = camera_look(perfect)
for title, image in [("ідеальний", perfect), ("camera_look", camera)]:
    print(title, "opencv:", DECODERS["opencv"](image), "zxing:", DECODERS["zxing"](image))

## Твої фото (коли будуть)

Поклади фото в `data/raw/own/` і заповни `labels.csv` (див. README).
TODO: порівняти частку прочитаних на твоїх фото при `normal` з синтетикою.

In [ ]:
own = load_own_photos()
own_rows = []
for sample in own:
    for name in decoders_for(sample["format"]):
        results = DECODERS[name](sample["image"])
        correct = False
        for result in results:
            if result["text"] == sample["text"]:
                correct = True
        own_rows.append({"light": sample["light"], "decoder": name, "correct": correct})
if len(own_rows) > 0:
    own_table = pd.DataFrame(own_rows)
    print(own_table.pivot_table(index="light", columns="decoder", values="correct", aggfunc="mean"))